In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ================================================
# CONFIG – ONLY these 4 machines
# ================================================
FILE_PATH = Path("Copy of Master Data_290102026 2 - Copy (wecompress.com).xlsx")
MASTER_SHEET   = "Master Data "
CATEGORY_SHEET = "Sheet1"

ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]

MAX_HOURS_PER_DAY    = 22.0
CHANGEOVER_HOURS     = 40 / 60.0
MAX_PARTS_PER_MACHINE = 3

# ================================================
# MACHINE NORMALIZATION FUNCTIONS
# ================================================
def normalize_machine(s):
    if pd.isna(s) or not str(s).strip():
        return None
    s = str(s).strip().upper()
    s = s.replace(".", "-").replace("M.P-", "MP-").replace("MP.", "MP-")
    s = s.replace(" ", "-")
    return s


def get_allowed_machines(cell):
    if pd.isna(cell):
        return []
    parts = str(cell).split(",")
    cleaned = [normalize_machine(x) for x in parts if normalize_machine(x)]
    return [m for m in set(cleaned) if m in ALLOWED_MACHINES]


# ================================================
# LOAD CATEGORY MAPPING
# ================================================
print("Loading categories from Sheet1...")
cat_df = pd.read_excel(FILE_PATH, sheet_name=CATEGORY_SHEET)
cat_df = cat_df[["PARTNO", "Category"]].dropna(subset=["PARTNO"])
cat_map = dict(zip(cat_df["PARTNO"], cat_df["Category"]))
print(f"→ {len(cat_map):,} category mappings loaded")


# ================================================
# AGGREGATE MASTER DATA
# ================================================
print("\nProcessing Master Data sheet...")
master = pd.read_excel(FILE_PATH, sheet_name=MASTER_SHEET)

for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

records = []
for child, g in master.groupby("Child Part"):
    daily_demand = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_demand <= 0:
        continue

    net_req = daily_demand + g["Minimum Quantity"].iloc[0] - g["Inventory_25"].iloc[0]
    if net_req <= 0:
        continue

    cycle_valid = g["Cycle Time"][g["Cycle Time"] > 0]
    if cycle_valid.empty:
        continue
    cycle_sec = cycle_valid.iloc[0]

    machines = get_allowed_machines(",".join(g["Vertical Machines"].dropna().astype(str)))
    if not machines:
        continue

    records.append({
        "Child Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_req,
        "Cycle_Time_sec": cycle_sec,
        "Eligible_Machines": machines,
        "Category": cat_map.get(child, "Unknown")
    })

df = pd.DataFrame(records)

# Safety: make sure Category column exists
if "Category" not in df.columns:
    df["Category"] = "Unknown"

print(f"\nValid parts with at least one of MP-01/05/10/17: {len(df):,}")
print("\nCategories:")
print(df["Category"].value_counts(dropna=False))

# Only Repeater + Stranger
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()
print(f"Parts to schedule: {len(to_schedule):,}")

if len(to_schedule) == 0:
    print("No Repeater or Stranger parts assigned to MP-01/05/10/17.")
    print("→ Check if these machines appear in 'Vertical Machines' column for Repeater/Stranger parts")
    # exit()   # comment out if you want to continue anyway

# ================================================
# SCHEDULER – only on the 4 machines
# ================================================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_sequence = {m: [] for m in ALLOWED_MACHINES}
schedule = []

print("\nPlanning only on MP-01, MP-05, MP-10, MP-17 ...")
to_schedule = to_schedule.sort_values("Net_Required", ascending=False)

for _, row in to_schedule.iterrows():
    hrs_per_pc = row["Cycle_Time_sec"] / 3600.0
    if hrs_per_pc <= 0:
        continue

    qty_left = row["Net_Required"]
    eligible = row["Eligible_Machines"]  # already filtered

    for m in sorted(eligible, key=lambda x: machine_load[x]):
        if len(machine_sequence[m]) >= MAX_PARTS_PER_MACHINE:
            continue

        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS:
            continue

        setup_h = CHANGEOVER_HOURS
        free_after = free - setup_h
        if free_after <= 0:
            continue

        max_qty = free_after / hrs_per_pc
        assign_qty = min(qty_left, max_qty)
        if assign_qty < 10:
            continue

        assign_h = assign_qty * hrs_per_pc

        machine_load[m] += assign_h + setup_h
        machine_sequence[m].append({
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        schedule.append({
            "Machine": m,
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        qty_left -= assign_qty
        if qty_left <= 0:
            break

# ================================================
# DISPLAY PLAN IN CONSOLE
# ================================================
print("\n" + "="*90)
print("FINAL PLAN – ONLY MP-01, MP-05, MP-10, MP-17")
print("="*90)

total_hours = sum(machine_load.values())

for m in ALLOWED_MACHINES:
    seq = machine_sequence.get(m, [])
    h = machine_load.get(m, 0.0)
    if h < 0.1 and not seq:
        print(f"\n{m} → No parts assigned")
        continue

    print(f"\n🛠 {m}   {h:6.1f} / 22.0 h   ({h/22*100:5.1f}%)   {len(seq)} parts")
    for p in seq:
        print(f"   • {p['Child Part']:22}   {p['Qty']:>7,} pcs   {p['Hours']:>5.1f}h   {p['Category']}")

print("\n" + "-"*90)
print(f"Total hours used : {total_hours:.1f} h")
print(f"Parts scheduled  : {len(schedule)}")
print("-"*90)